# CS26012 — Crop Disease Model Training (TensorFlow / Keras, MobileNetV2)

| Part | What it does |
|---|---|
| A | Setup: Drive, repo, libraries, settings |
| B | Explore the dataset: classes, counts, corrupt files, image sizes *(your cells)* |
| C | Duplicate check: exact (MD5) + near-duplicate (perceptual hash) *(your cells)* |
| D | **Clean**: remove cross-class label conflicts + greedy near-duplicate removal |
| E | **Split** 70/15/15 (stratified, seed 42) → save a manifest → copy to local disk |
| F | `tf.data` pipelines + augmentation + class weights |
| G | Build MobileNetV2 (frozen backbone, new head) and train |
| H | Test-set evaluation (in-distribution) |
| I | Save model + metadata to Drive |
| J | Reload check + out-of-distribution (OOD) check on your own photos |

Runtime → Change runtime type → **T4 GPU**.

## A. Setup
`drive.mount` makes Google Drive appear as a normal folder at `/content/drive/MyDrive` (this cell was missing — without it the dataset path only works if Drive was mounted from the sidebar).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/Jugal1406/CS26012-Crop_Disease_Detection.git

In [ ]:
%cd CS26012-Crop_Disease_Detection

In [ ]:
!ls

In [ ]:
!ls models

In [ ]:
!ls notebooks

In [ ]:
!pip install -q tensorflow scikit-learn matplotlib seaborn pillow imagehash

In [ ]:
import os
import json
import random
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow Version :", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

### Settings
Everything you might change lives in this one cell. `CROP` must match the value the frontend sends (`chili`, `potato`, `groundnut`) — the backend uses it to pick the model file.

In [ ]:
CROP = 'chili'

DRIVE_DATA_ROOT = '/content/drive/MyDrive/Crop_Disease_Dataset'
CROP_FOLDERS = {
    'chili':     'Chili Leaf Disease Original Dataset',
    'potato':    'Potato',
    'groundnut': 'Groundnut',
}
LOCAL_DATA_DIR = Path('/content/data') / CROP            # fast local copy (wiped when the session ends)
MODEL_SAVE_DIR = Path('/content/drive/MyDrive/CS26012_models')

SEED          = 42
SPLIT         = (0.70, 0.15, 0.15)   # train / valid / test
DUP_THRESHOLD = 5                    # pHash Hamming distance <= this counts as a near-duplicate
IMG_SIZE      = 224
BATCH_SIZE    = 32
HEAD_EPOCHS   = 15                   # phase 1: backbone frozen (early stopping may end it sooner)
HEAD_LR       = 1e-3
FINETUNE_EPOCHS      = 0             # phase 2 (optional experiment): unfreeze the top of the backbone
FINETUNE_LR          = 1e-5
FINETUNE_LAST_LAYERS = 30

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

## B. Explore the dataset

In [ ]:
DATASET_DIR = Path(DRIVE_DATA_ROOT) / CROP_FOLDERS[CROP]

print("Dataset exists:", DATASET_DIR.exists())
print("Dataset path:", DATASET_DIR)

In [ ]:
for item in DATASET_DIR.iterdir():
    print(item.name, "->", "Folder" if item.is_dir() else "File")

In [ ]:
class_dirs = sorted([
    folder for folder in DATASET_DIR.iterdir()
    if folder.is_dir()
])

print("Number of classes:", len(class_dirs))

print("\nClasses:")
for i, folder in enumerate(class_dirs):
    print(i, "->", folder.name)

In [ ]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

class_counts = {}

for class_dir in class_dirs:
    count = sum(
        1
        for file in class_dir.rglob("*")
        if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS
    )

    class_counts[class_dir.name] = count

class_counts

In [ ]:
class_df = pd.DataFrame(
    list(class_counts.items()),
    columns=["Class", "Image_Count"]
)

class_df = class_df.sort_values(
    "Image_Count",
    ascending=False
).reset_index(drop=True)

class_df

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=class_df,
    x="Image_Count",
    y="Class"
)

plt.title("Chili Dataset Class Distribution")
plt.xlabel("Number of Images")
plt.ylabel("Disease Class")
plt.show()

In [ ]:
corrupted_images = []

for class_dir in class_dirs:
    for image_path in class_dir.rglob("*"):

        if not image_path.is_file():
            continue

        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        try:
            with Image.open(image_path) as img:
                img.verify()

        except Exception:
            corrupted_images.append(str(image_path))

print("Corrupted images:", len(corrupted_images))

for image in corrupted_images[:20]:
    print(image)

In [ ]:
image_info = []

for class_dir in class_dirs:
    for image_path in class_dir.rglob("*"):

        if not image_path.is_file():
            continue

        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        try:
            with Image.open(image_path) as img:
                image_info.append({
                    "class": class_dir.name,
                    "filename": image_path.name,
                    "width": img.width,
                    "height": img.height,
                    "mode": img.mode
                })

        except:
            pass

image_info_df = pd.DataFrame(image_info)

print("Images inspected:", len(image_info_df))

image_info_df.head()

In [ ]:
image_info_df[["width", "height", "mode"]].describe(include="all")

## C. Duplicate check
**Exact duplicates (MD5):** identical files. Note the result — several *Bacterial Spot* images are byte-for-byte identical to *Cercospora Leaf Spot* images. The same photo carries two different labels, so the model is being taught a contradiction.

In [ ]:
def get_file_hash(file_path):
    md5 = hashlib.md5()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            md5.update(chunk)

    return md5.hexdigest()

In [ ]:
hash_records = []

for class_dir in class_dirs:
    for image_path in class_dir.rglob("*"):

        if not image_path.is_file():
            continue

        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        try:
            file_hash = get_file_hash(image_path)

            hash_records.append({
                "class": class_dir.name,
                "path": str(image_path),
                "hash": file_hash
            })

        except:
            pass

hash_df = pd.DataFrame(hash_records)

print("Total images hashed:", len(hash_df))

In [ ]:
duplicate_groups = (
    hash_df
    .groupby("hash")
    .filter(lambda x: len(x) > 1)
    .sort_values("hash")
)

print(
    "Images involved in duplicate groups:",
    len(duplicate_groups)
)

In [ ]:
for image_hash, group in duplicate_groups.groupby("hash"):
    print("\n" + "=" * 80)
    print("Duplicate group:")

    for _, row in group.iterrows():
        print(f"Class: {row['class']}")
        print(f"Path : {row['path']}")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

duplicate_hashes = duplicate_groups["hash"].unique()

for i, image_hash in enumerate(duplicate_hashes):

    group = duplicate_groups[
        duplicate_groups["hash"] == image_hash
    ]

    plt.figure(figsize=(12, 4))

    for j, (_, row) in enumerate(group.iterrows()):
        img = Image.open(row["path"])

        plt.subplot(1, len(group), j + 1)
        plt.imshow(img)
        plt.title(row["class"])
        plt.axis("off")

    plt.suptitle(f"Duplicate Group {i + 1}")
    plt.show()

**Near-duplicates (perceptual hash):** pHash turns each image into a 64-bit fingerprint of its overall structure; the *Hamming distance* (how many bits differ) between two fingerprints is small when images look alike, even if resized/re-compressed/slightly edited.

The original pairwise loop called `df.iloc[...]` ~1.7 million times (slow). The version below gives the **same pairs** but uses numpy to compare every fingerprint against every other in one go, and also tells us whether each pair is **within one class** or **across two classes**.

In [ ]:
!pip install imagehash

In [ ]:
import imagehash

records = []

for class_name in sorted(os.listdir(DATASET_DIR)):

    class_path = os.path.join(DATASET_DIR, class_name)

    if not os.path.isdir(class_path):
        continue

    for filename in sorted(os.listdir(class_path)):

        file_path = os.path.join(class_path, filename)
        if Path(filename).suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        try:
            img = Image.open(file_path).convert("RGB")

            # Perceptual hash
            phash = imagehash.phash(img)

            records.append({
                "class": class_name,
                "filename": filename,
                "path": file_path,
                "phash": str(phash)
            })

        except Exception:
            pass

df_hash = pd.DataFrame(records)

print("Total images processed:", len(df_hash))

In [ ]:
def hashes_to_uint64(hex_hashes):
    # '8f3a...'(16 hex chars) -> one 64-bit integer, so numpy can compare them fast
    return np.array([int(h, 16) for h in hex_hashes], dtype=np.uint64)

def hamming_matrix(h):
    # distance[i, j] = number of differing bits between fingerprint i and j
    return np.bitwise_count(h[:, None] ^ h[None, :]).astype(np.uint8)

def near_duplicate_pairs(df, threshold):
    h = hashes_to_uint64(df['phash'])
    dist = hamming_matrix(h)
    i, j = np.where(np.triu(dist <= threshold, k=1))       # upper triangle: each pair once, no self-pairs
    pairs = pd.DataFrame({
        'image1_class': df['class'].to_numpy(dtype=object)[i], 'image1': df['filename'].to_numpy(dtype=object)[i], 'image1_path': df['path'].to_numpy(dtype=object)[i],
        'image2_class': df['class'].to_numpy(dtype=object)[j], 'image2': df['filename'].to_numpy(dtype=object)[j], 'image2_path': df['path'].to_numpy(dtype=object)[j],
        'hash_distance': dist[i, j],
    })
    pairs['same_class'] = pairs['image1_class'] == pairs['image2_class']
    return pairs

In [ ]:
near_duplicates_df = near_duplicate_pairs(df_hash, DUP_THRESHOLD)
print('Near-duplicate pairs found:', len(near_duplicates_df))
print('  within the same class :', near_duplicates_df['same_class'].sum())
print('  ACROSS two classes    :', (~near_duplicates_df['same_class']).sum(), ' <- label conflicts')

print('\nPairs by distance (0 = identical fingerprint):')
print(near_duplicates_df['hash_distance'].value_counts().sort_index().to_string())

print('\nCross-class pairs by class combination:')
cross = near_duplicates_df[~near_duplicates_df['same_class']]
print(cross.groupby(['image1_class', 'image2_class']).size().to_string() if len(cross) else '  none')

**Look before you trust the threshold.** The cell below shows random pairs at each distance. If pairs at distance 5 are clearly *different* leaves, lower `DUP_THRESHOLD` in Settings and re-run from there; if pairs at 5 are still obviously the same photo, you could raise it.

In [ ]:
def show_pairs(pairs, distance, n=3):
    sub = pairs[pairs['hash_distance'] == distance]
    if len(sub) == 0:
        print(f'No pairs at distance {distance}'); return
    sub = sub.sample(min(n, len(sub)), random_state=SEED)
    fig, axes = plt.subplots(len(sub), 2, figsize=(7, 3.3 * len(sub)), squeeze=False)
    for row, (_, p) in enumerate(sub.iterrows()):
        for col, side in enumerate(('image1', 'image2')):
            axes[row][col].imshow(Image.open(p[f'{side}_path']))
            axes[row][col].set_title(f"{p[f'{side}_class']}\n{p[side]}", fontsize=8)
            axes[row][col].axis('off')
    fig.suptitle(f'Hamming distance = {distance}'); plt.tight_layout(); plt.show()

for d in sorted(near_duplicates_df['hash_distance'].unique()):
    show_pairs(near_duplicates_df, d)

## D. Clean the dataset

Two rules, applied in this order:

1. **Cross-class conflicts → remove every copy.** If (near-)identical images sit in two different classes, we can't know which label is right, so none of them are trusted. (Keeping one at random would be guessing the diagnosis.)
2. **Within-class near-duplicates → keep one, drop the rest — using the *greedy* method.**
   Images are walked through in sorted filename order. Each image is compared **only against images already kept**. If it's within `DUP_THRESHOLD` of a kept image it's dropped; otherwise it becomes a new kept reference.

   ⚠️ This is the method validated earlier in the project. We deliberately do **not** group images into connected clusters (A~B, B~C ⇒ A,B,C merged): that "chaining" wrongly merged dissimilar images at scale.

Nothing is deleted from Drive — we only build a list (`clean_df`) of which files to use, and save the removed ones to a CSV so the decisions are documented.

In [ ]:
def find_cross_class_conflicts(df, threshold):
    # Indices of images that have a near-duplicate in a DIFFERENT class.
    h = hashes_to_uint64(df['phash'])
    dist = hamming_matrix(h)
    labels = df['class'].to_numpy(dtype=object)   # plain numpy array (works on pandas 2 and 3)
    different_class = labels[:, None] != labels[None, :]
    conflict = ((dist <= threshold) & different_class).any(axis=1)
    return set(np.where(conflict)[0])

def greedy_dedup(df, threshold):
    # Returns (kept_indices, {dropped_index: index_of_the_kept_image_it_duplicates}).
    # Works class by class; each image is compared only with already-kept images.
    kept, dropped = [], {}
    for _, group in df.groupby('class', sort=True):
        group = group.sort_values('filename')
        ref_idx, ref_hash = [], np.empty(0, dtype=np.uint64)
        for idx, hx in zip(group.index, hashes_to_uint64(group['phash'])):
            if len(ref_hash):
                d = np.bitwise_count(ref_hash ^ hx)
                nearest = int(np.argmin(d))
                if d[nearest] <= threshold:
                    dropped[idx] = ref_idx[nearest]
                    continue
            ref_idx.append(idx)
            ref_hash = np.append(ref_hash, hx)
        kept.extend(ref_idx)
    return kept, dropped

def clean_dataset(df, threshold):
    df = df.reset_index(drop=True)
    conflicts = find_cross_class_conflicts(df, threshold)
    no_conflict = df.drop(index=list(conflicts))
    kept, dropped = greedy_dedup(no_conflict, threshold)
    clean = df.loc[sorted(kept)].reset_index(drop=True)
    removed = pd.concat([
        df.loc[sorted(conflicts)].assign(reason='cross-class conflict', duplicate_of=''),
        df.loc[list(dropped)].assign(reason='within-class near-duplicate',
                                     duplicate_of=[df.loc[r, 'filename'] for r in dropped.values()]),
    ], ignore_index=True)
    return clean, removed

In [ ]:
clean_df, removed_df = clean_dataset(df_hash, DUP_THRESHOLD)

summary = pd.DataFrame({
    'original': df_hash['class'].value_counts(),
    'cross_class_removed': removed_df[removed_df.reason == 'cross-class conflict']['class'].value_counts(),
    'near_dup_removed': removed_df[removed_df.reason == 'within-class near-duplicate']['class'].value_counts(),
    'clean': clean_df['class'].value_counts(),
}).fillna(0).astype(int).sort_index()
print(summary.to_string())
print(f"\nTotal: {len(df_hash)} -> {len(clean_df)} images ({len(removed_df)} removed)")
print(f"Max:min ratio after cleaning = {summary['clean'].max() / summary['clean'].min():.1f}:1 (policy target <= 5:1)")
for c, n in summary['clean'].items():
    if n < 100: print(f'  WARNING: "{c}" has only {n} images (hard floor is 100)')

MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
removed_df.to_csv(MODEL_SAVE_DIR / f'{CROP}_removed_images.csv', index=False)
print('Removed-image log saved to', MODEL_SAVE_DIR / f'{CROP}_removed_images.csv')

## E. Split 70 / 15 / 15 and copy locally

* **Stratified**: each class is split separately, so every class keeps the same proportions in train/valid/test.
* **Reproducible**: files are sorted, then shuffled with `random.Random(42)` → the same split every time, on any machine.
* Splitting happens **after** de-duplication, so near-copies can no longer leak from train into test (which would inflate the test score).
* The split is saved as a **manifest CSV** in Drive — your exact train/valid/test lists for the report.
* Files are then **copied** (not symlinked) to Colab's local disk, which is much faster to read during training than Drive.

In [ ]:
import shutil

def stratified_split(df, fractions=(0.7, 0.15, 0.15), seed=42):
    rng = random.Random(seed)
    parts = []
    for cls, group in df.groupby('class', sort=True):
        paths = sorted(group['path'])
        rng.shuffle(paths)
        n = len(paths)
        n_valid, n_test = int(n * fractions[1]), int(n * fractions[2])
        n_train = n - n_valid - n_test                   # rounding leftovers go to train
        for split, chunk in (('train', paths[:n_train]),
                             ('valid', paths[n_train:n_train + n_valid]),
                             ('test',  paths[n_train + n_valid:])):
            parts.append(pd.DataFrame({'path': chunk, 'class': cls, 'split': split}))
    return pd.concat(parts, ignore_index=True)

def copy_split(manifest, dest_root):
    dest_root = Path(dest_root)
    if dest_root.exists():
        shutil.rmtree(dest_root)          # start clean so re-runs never mix old + new files
    for path, cls, split in zip(manifest['path'], manifest['class'], manifest['split']):
        target = dest_root / split / cls
        target.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target / Path(path).name)

In [ ]:
manifest = stratified_split(clean_df, SPLIT, SEED)
manifest.to_csv(MODEL_SAVE_DIR / f'{CROP}_split_manifest.csv', index=False)
print(manifest.pivot_table(index='class', columns='split', values='path', aggfunc='count')[['train', 'valid', 'test']])

copy_split(manifest, LOCAL_DATA_DIR)
print('\nCopied to', LOCAL_DATA_DIR)

## F. Input pipeline, augmentation, class weights

**`image_dataset_from_directory`** reads `train/<class>/*.jpg` and gives back batches of `(images, labels)`. Labels are numbers assigned in the order of `CLASS_NAMES` (alphabetical) → `CLASS_NAMES[label]` turns a number back into a disease name.
* `shuffle=True` for train only. Valid/test are **not** shuffled, so predictions line up with labels during evaluation.
* `.cache()` keeps decoded images in RAM after the first epoch; `.prefetch()` prepares the next batch while the GPU is busy.

**Augmentation** (random flips, rotation, zoom, contrast, brightness) gives the model a slightly different version of each image every epoch → more field-like variety and less overfitting. These are Keras *layers* inside the model: they only act during `fit()` and switch themselves off automatically for `predict()`/`evaluate()`.

**Class weights** (Keras's way of handling imbalance): the loss from a rare class counts more. Weight = `total / (num_classes × class_count)`, so a class with half the average number of images counts double. This replaces the "weighted sampling" used in the PyTorch plan — same goal, no images thrown away.

In [ ]:
CLASS_NAMES = sorted(clean_df['class'].unique())
NUM_CLASSES = len(CLASS_NAMES)
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(split, shuffle):
    return keras.utils.image_dataset_from_directory(
        str(LOCAL_DATA_DIR / split),
        labels='inferred', label_mode='int', class_names=CLASS_NAMES,
        image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
        shuffle=shuffle, seed=SEED,
    )

train_ds = make_ds('train', shuffle=True).cache().prefetch(AUTOTUNE)
valid_ds = make_ds('valid', shuffle=False).cache().prefetch(AUTOTUNE)
test_ds  = make_ds('test',  shuffle=False).cache().prefetch(AUTOTUNE)

train_counts = manifest[manifest.split == 'train']['class'].value_counts()
class_weight = {i: len(manifest[manifest.split == 'train']) / (NUM_CLASSES * train_counts[c])
                for i, c in enumerate(CLASS_NAMES)}
for i, c in enumerate(CLASS_NAMES):
    print(f'{i}  {c:<24} train={train_counts[c]:<4} weight={class_weight[i]:.2f}')

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.08),          # up to ±0.08 of a full turn ≈ ±29°
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),         # works on 0-255 pixels, so it goes BEFORE rescaling
], name='augmentation')

# Preview: one training image, augmented 8 different ways
for images, labels in train_ds.take(1):
    plt.figure(figsize=(10, 5))
    for k in range(8):
        aug = data_augmentation(images[:1], training=True)
        plt.subplot(2, 4, k + 1); plt.imshow(aug[0].numpy().astype('uint8')); plt.axis('off')
    plt.suptitle(f'Augmentations of one "{CLASS_NAMES[labels[0]]}" image'); plt.show()

## G. Build the model and train

```
raw image (0-255) → augmentation → rescale to [-1, 1] → MobileNetV2 backbone (frozen) → average pool → dropout → Dense(softmax)
```
* **Rescaling(1/127.5, offset=-1)** does exactly what `mobilenet_v2.preprocess_input` does (pixels 0–255 → −1…1). Because it is *inside* the model, the backend can feed plain 0–255 RGB images and can't get the preprocessing wrong. (A `Rescaling` layer also saves/loads cleanly, unlike a `Lambda` wrapping `preprocess_input`.)
* **`base.trainable = False`** freezes the 2.2 M pretrained backbone weights; only the new head learns.
* **`base(x, training=False)`** keeps the backbone's BatchNorm layers in inference mode. Without it, BatchNorm would re-estimate its statistics from our small dataset and wreck the pretrained features — the most common Keras transfer-learning bug, especially once you fine-tune.
* **GlobalAveragePooling2D** turns the backbone's 7×7×1280 feature map into 1280 numbers; **Dropout(0.2)** randomly hides 20 % of them during training to reduce overfitting; **Dense(softmax)** outputs one probability per class.

**Callbacks** run at the end of each epoch:
* `ModelCheckpoint` saves the model whenever validation accuracy improves → we always keep the **best** epoch, not the last.
* `EarlyStopping` stops if validation accuracy hasn't improved for 5 epochs.

In [ ]:
def build_model():
    base = keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                                          include_top=False, weights='imagenet')
    base.trainable = False
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image_0_255')
    x = data_augmentation(inputs)
    x = layers.Rescaling(1 / 127.5, offset=-1, name='to_minus1_plus1')(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='disease_probs')(x)
    return keras.Model(inputs, outputs, name=f'{CROP}_mobilenetv2'), base

model, base = build_model()
model.compile(optimizer=keras.optimizers.Adam(HEAD_LR),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary(show_trainable=True)

In [ ]:
import time
BEST_PATH = f'/content/{CROP}_best.keras'
checkpoint_cb = keras.callbacks.ModelCheckpoint(BEST_PATH, monitor='val_accuracy',
                                                save_best_only=True, verbose=1)

start = time.time()
history = model.fit(train_ds, validation_data=valid_ds, epochs=HEAD_EPOCHS,
                    class_weight=class_weight,
                    callbacks=[checkpoint_cb,
                               keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5)])
hist = {k: list(v) for k, v in history.history.items()}

if FINETUNE_EPOCHS > 0:
    # Unfreeze only the top FINETUNE_LAST_LAYERS layers of the backbone. BatchNorm stays in
    # inference mode because the backbone is still called with training=False.
    base.trainable = True
    for layer in base.layers[:-FINETUNE_LAST_LAYERS]:
        layer.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(FINETUNE_LR),   # must re-compile after changing trainable
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    ft = model.fit(train_ds, validation_data=valid_ds,
                   epochs=len(hist['loss']) + FINETUNE_EPOCHS, initial_epoch=len(hist['loss']),
                   class_weight=class_weight,
                   callbacks=[checkpoint_cb,          # same instance -> still remembers the best so far
                              keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5)])
    for k in hist: hist[k] += ft.history.get(k, [])

train_time = time.time() - start
best_epoch = int(np.argmax(hist['val_accuracy'])) + 1
print(f"\nBest val accuracy {max(hist['val_accuracy']):.3f} at epoch {best_epoch} | {train_time:.0f}s")

# Load the best checkpoint (this also proves the saved file loads correctly)
model = keras.models.load_model(BEST_PATH)

In [ ]:
ep = range(1, len(hist['loss']) + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(ep, hist['loss'], label='train'); ax[0].plot(ep, hist['val_loss'], label='valid')
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(ep, hist['accuracy'], label='train'); ax[1].plot(ep, hist['val_accuracy'], label='valid')
ax[1].set_title('Accuracy'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.show()

## H. Test-set evaluation (in-distribution)
The test set was never used for training *or* for picking the best epoch → this is the honest held-out score.
* **Classification report** — per class: *precision* (when it says "Curl Virus", how often is it right?), *recall* (of all real Curl Virus leaves, how many did it catch?), *F1* (balance of both).
* **Confusion matrix** — rows = true class, columns = predicted. Watch the Bacterial Spot ↔ Cercospora cell in particular, given the duplicates found earlier.

⚠️ This number is **in-distribution only** (same source, same camera conditions). Part J measures the other side.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = model.predict(test_ds, verbose=0)
y_pred = y_prob.argmax(axis=1)
test_acc = float((y_pred == y_true).mean())
print(f'TEST accuracy: {test_acc:.3f}  ({len(y_true)} images)\n')
print(classification_report(y_true, y_pred, labels=range(NUM_CLASSES), target_names=CLASS_NAMES, digits=3, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, xticks_rotation=45, cmap='Greens', colorbar=False)
plt.title(f'{CROP} — test confusion matrix'); plt.tight_layout(); plt.show()

## I. Save to Drive
Files written to `MyDrive/CS26012_models/`:
* `<crop>_mobilenetv2.keras` — the full model (architecture + weights + the rescaling step). Download into the repo's `models/` folder.
* `<crop>_mobilenetv2_meta.json` — what the **backend** needs: class names in label order, image size, and how to prepare an image.
* `<crop>_mobilenetv2_metrics.json` — numbers for the report/README.
* (already saved earlier) `<crop>_removed_images.csv`, `<crop>_split_manifest.csv`.

In [ ]:
import datetime
stem = f'{CROP}_mobilenetv2'
model.save(str(MODEL_SAVE_DIR / f'{stem}.keras'))

meta = {
    'crop': CROP, 'arch': 'MobileNetV2', 'framework': f'tensorflow {tf.__version__}',
    'class_names': CLASS_NAMES, 'img_size': IMG_SIZE,
    'input': 'RGB, float32, pixel values 0-255, shape (1, 224, 224, 3); the model rescales internally',
    'resize': 'center-crop to a square, then resize to img_size (bilinear)',
}
(MODEL_SAVE_DIR / f'{stem}_meta.json').write_text(json.dumps(meta, indent=2))

metrics = {
    **meta, 'trained_at': datetime.datetime.now().isoformat(timespec='seconds'),
    'seed': SEED, 'split': SPLIT, 'dup_threshold': DUP_THRESHOLD, 'batch_size': BATCH_SIZE,
    'head_epochs': HEAD_EPOCHS, 'head_lr': HEAD_LR, 'finetune_epochs': FINETUNE_EPOCHS, 'finetune_lr': FINETUNE_LR,
    'images_original': len(df_hash), 'images_clean': len(clean_df), 'images_removed': len(removed_df),
    'split_counts': manifest.groupby(['split', 'class']).size().unstack(fill_value=0).to_dict(orient='index'),
    'class_weight': {CLASS_NAMES[i]: round(w, 3) for i, w in class_weight.items()},
    'best_epoch': best_epoch, 'best_val_acc': float(max(hist['val_accuracy'])), 'test_acc': test_acc,
    'train_time_s': round(train_time, 1), 'history': {k: [float(v) for v in vals] for k, vals in hist.items()},
    'test_report': classification_report(y_true, y_pred, labels=range(NUM_CLASSES), target_names=CLASS_NAMES,
                                         output_dict=True, zero_division=0),
    'confusion_matrix': cm.tolist(),
}
to_native = lambda o: o.item() if hasattr(o, 'item') else str(o)   # numpy numbers -> plain Python for JSON
(MODEL_SAVE_DIR / f'{stem}_metrics.json').write_text(json.dumps(metrics, indent=2, default=to_native))

size_mb = (MODEL_SAVE_DIR / f'{stem}.keras').stat().st_size / 1e6
print(f'Saved {stem}.keras ({size_mb:.1f} MB), {stem}_meta.json, {stem}_metrics.json -> {MODEL_SAVE_DIR}')

## J. Reload check + out-of-distribution (OOD) check

`load_for_inference` + `predict_image` are written exactly the way the FastAPI backend will use them: load the `.keras` file and its meta JSON, then for any photo → RGB → center-crop to square → resize to 224 → predict.

**Center-crop first** because phone photos are rarely square; squashing a 4:3 photo would distort the leaf (the training images were all 1000×1000, so they were never squashed).

The reload check confirms the saved file gives the same predictions as the in-memory model. Then upload a few leaf photos **from outside the dataset** (your own phone, Google Images, a farm visit) — report these next to the test accuracy; it's the honest measure of how the app will behave for a real farmer.

In [ ]:
def load_for_inference(stem_path):
    m = keras.models.load_model(f'{stem_path}.keras')
    info = json.loads(Path(f'{stem_path}_meta.json').read_text())
    return m, info

def prepare_image(img, size):
    img = img.convert('RGB')
    w, h = img.size
    side = min(w, h)
    left, top = (w - side) // 2, (h - side) // 2
    img = img.crop((left, top, left + side, top + side)).resize((size, size), Image.BILINEAR)
    return np.asarray(img, dtype=np.float32)[None, ...]     # shape (1, size, size, 3), values 0-255

def predict_image(m, info, img, top_k=3):
    probs = m.predict(prepare_image(img, info['img_size']), verbose=0)[0]
    order = probs.argsort()[::-1][:top_k]
    return [(info['class_names'][i], round(float(probs[i]) * 100, 1)) for i in order]

reloaded, info = load_for_inference(MODEL_SAVE_DIR / stem)
same = np.allclose(  # same weights -> same outputs
    reloaded.predict(test_ds, verbose=0), y_prob, atol=1e-5)
print('Reloaded model matches in-memory predictions:', same)
sample = manifest[manifest.split == 'test'].iloc[0]
print(f"Example: true = {sample['class']} | predicted = {predict_image(reloaded, info, Image.open(sample['path']))}")

In [ ]:
from google.colab import files
uploaded = files.upload()          # pick one or more outside-the-dataset leaf photos
for name in uploaded:
    print(f'{name}: {predict_image(reloaded, info, Image.open(name))}')